# TASK4 — 海龟交易策略 (Turtle Trading Strategy)**作者：王艳鹏**  **日期：2026-07-11**## 任务目标1. 了解海龟策略的核心思想与关键优势2. 掌握 Donchian 通道、ATR、止损条件等核心概念3. Python 实现完整策略：数据加载 → 信号生成 → 可视化 → 回测4. 多股票、多参数对比实验，总结适应场景

## 1. 海龟策略理论基础### 1.1 核心思想海龟交易法则由 Richard Dennis 和 William Eckhardt 于 1980 年代提出，是一套**完全机械化**的趋势跟踪交易系统。**五大核心要素：**| 要素 | 内容 ||------|------|| 要素一：入场规则 | 价格突破 N 日高点 → 顺势买入 || 要素二：止损规则 | 入场价 - 2×ATR 硬性止损 || 要素三：仓位管理 | Unit = (资金 × 1%) / ATR，波动自适应 || 要素四：止盈离场 | 反向突破通道 → 让利润奔跑 || 要素五：风险控制 | 单市场≤4 Unit、同向≤12 Unit、总风险≤12% |### 1.2 关键优势- **规则明确、可回测、可复现** — 所有决策由数据驱动- **截断亏损、让利润奔跑** — 止损控制下行风险，趋势跟踪捕获大行情- **波动自适应仓位管理** — 大波动少买、小波动多买- **跨市场通用** — 同一套规则适用于股票、期货、外汇### 1.3 双系统- **系统一（中短线）**：20 日突破入场，10 日跌破退出- **系统二（中长线）**：55 日突破入场，20 日跌破退出

## 2. 核心概念详解### 2.1 Donchian 高低点通道- **上轨** = N 日最高价的最大值- **下轨** = N 日最低价的最小值- **中轨** = (上轨+下轨) / 2### 2.2 ATR (Average True Range)$$TR = \max(H_t - L_t, |H_t - C_{t-1}|, |L_t - C_{t-1}|)$$$$ATR = MA(TR, 14)$$### 2.3 单位 (Unit) 仓位管理$$Unit = \lfloor \frac{账户资金 \times 1\%}{ATR} \rfloor$$### 2.4 金字塔加仓浮盈每增加 0.5×ATR，加仓 1 个单位（最多 4 个）。### 2.5 止损条件- ATR 止损：价格 ≤ 入场价 - 2×ATR- 通道退出：价格跌破 M 日最低价

## 3. Python 实现

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport warningswarnings.filterwarnings('ignore')plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Heiti SC', 'SimHei']plt.rcParams['axes.unicode_minus'] = False# ── 1) 加载数据 ──def load_data(symbol):    path = f"../TASK3/data/standard/{symbol}/1d.parquet"    df = pd.read_parquet(path)    df['trade_date'] = pd.to_datetime(df['trade_date'])    df = df.sort_values('trade_date').reset_index(drop=True)    return df# 测试加载比亚迪 A 股df = load_data('002594.SZ')print(f"数据形状: {df.shape}")print(f"日期范围: {df['trade_date'].min()} ~ {df['trade_date'].max()}")df.head()

In [ ]:
# ── 2) 计算 Donchian 通道 ──def calc_donchian(df, channel_period=20, exit_period=10):    df = df.copy()    df['upper_channel'] = df['high'].rolling(channel_period).max()    df['lower_channel'] = df['low'].rolling(channel_period).min()    df['mid_channel'] = (df['upper_channel'] + df['lower_channel']) / 2    df['exit_lower'] = df['low'].rolling(exit_period).min()    return dfdf = calc_donchian(df)# 可视化fig, ax = plt.subplots(figsize=(14, 6))recent = df.iloc[-252:]ax.plot(recent['trade_date'], recent['close'], 'black', lw=0.8, label='收盘价')ax.plot(recent['trade_date'], recent['upper_channel'], 'red', ls='--', lw=0.8, label='上轨(20日)')ax.plot(recent['trade_date'], recent['lower_channel'], 'green', ls='--', lw=0.8, label='下轨(20日)')ax.plot(recent['trade_date'], recent['mid_channel'], 'gray', ls=':', lw=0.5, label='中轨')ax.legend(); ax.grid(True, alpha=0.3)ax.set_title('图: Donchian 通道 (比亚迪A, 最近1年)', fontsize=12)plt.tight_layout(); plt.show()

In [ ]:
# ── 3) 计算 ATR ──def calc_atr(df, period=14):    df = df.copy()    df['prev_close'] = df['close'].shift(1)    df['tr1'] = df['high'] - df['low']    df['tr2'] = abs(df['high'] - df['prev_close'])    df['tr3'] = abs(df['low'] - df['prev_close'])    df['tr'] = df[['tr1', 'tr2', 'tr3']].max(axis=1)    df['atr'] = df['tr'].rolling(period).mean()    return dfdf = calc_atr(df)fig, ax1 = plt.subplots(figsize=(14, 4))recent = df.iloc[-252:]ax1.fill_between(recent['trade_date'], 0, recent['atr'], color='blue', alpha=0.3)ax1.plot(recent['trade_date'], recent['atr'], 'blue', lw=0.8, label='ATR(14)')ax1.set_title('图: ATR 波动率指标 (比亚迪A, 最近1年)', fontsize=12)ax1.legend(); ax1.grid(True, alpha=0.3)plt.tight_layout(); plt.show()# 计算 Unitrisk_amount = 100000 * 0.01recent['unit'] = np.floor(risk_amount / recent['atr']).fillna(0).astype(int)print(f"当前 ATR: {recent['atr'].iloc[-1]:.2f}")print(f"每个 Unit 股数: {recent['unit'].iloc[-1]}")print(f"每 Unit 风险: {recent['unit'].iloc[-1] * recent['atr'].iloc[-1]:.0f} 元")

In [ ]:
# ── 4) 策略回测引擎 ──def run_backtest(df, channel_period=20, exit_period=10, atr_period=14,                 account_value=100000, risk_pct=0.01, max_units=4):        df = calc_donchian(df, channel_period, exit_period)    df = calc_atr(df, atr_period)        risk_amount = account_value * risk_pct    df['unit_size'] = np.floor(risk_amount / df['atr']).fillna(0).astype(int)        cash = account_value    shares = 0    units_held = 0    entry_price = 0    positions = []    trades = []        warmup = max(channel_period, atr_period)        for i in range(warmup, len(df)):        row = df.iloc[i]        price, atr_val = row['close'], row['atr']        upper_prev = df.iloc[i-1]['upper_channel']        exit_lower_prev = df.iloc[i-1]['exit_lower']        unit_size = row['unit_size']                if pd.isna(upper_prev) or pd.isna(atr_val) or unit_size <= 0:            positions.append(0)            continue                signal = 0        if units_held > 0:            # 止损检查            stop = entry_price - 2 * atr_val            if row['low'] <= stop:                cash += shares * min(row['open'], stop)                trades.append({'date': row['trade_date'], 'type': '止损'})                shares, units_held, entry_price = 0, 0, 0                signal = -1            elif price < exit_lower_prev:                cash += shares * price                trades.append({'date': row['trade_date'], 'type': '通道退出'})                shares, units_held, entry_price = 0, 0, 0                signal = -1            elif units_held < max_units and price >= entry_price + 0.5 * atr_val * units_held:                add = int(risk_amount / atr_val)                if cash >= add * price:                    shares += add; cash -= add * price; units_held += 1                    trades.append({'date': row['trade_date'], 'type': f'加仓({units_held})'})                    signal = 1        else:            if price > upper_prev:                buy = unit_size                if cash >= buy * price:                    shares = buy; cash -= buy * price                    units_held = 1; entry_price = price                    trades.append({'date': row['trade_date'], 'type': '入场'})                    signal = 1                positions.append(signal)        df['signal'] = [0]*warmup + positions    df['nav'] = [account_value]*warmup +                 [(cash + shares * df.iloc[i]['close']) for i in range(warmup, len(df))]    df['nav'] = df['nav'].fillna(account_value)    return df, trades# 运行回测df_result, trades = run_backtest(df)print(f"总交易次数: {len(trades)}")for t in trades[:10]:    print(f"  {t['date'].date()}  {t['type']}")

In [ ]:
# ── 5) 策略信号可视化 ──fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10),                                gridspec_kw={'height_ratios': [3, 1]})fig.suptitle('图: 比亚迪A 海龟策略 (系统一: 20/10)', fontsize=14, fontweight='bold')recent = df_result.iloc[-504:]dates = recent['trade_date']ax1.plot(dates, recent['close'], 'black', lw=0.8, label='收盘价')ax1.plot(dates, recent['upper_channel'], 'red', ls='--', lw=0.8, label='上轨')ax1.plot(dates, recent['lower_channel'], 'green', ls='--', lw=0.8, label='下轨')buys = recent[recent['signal'] == 1]sells = recent[recent['signal'] == -1]if len(buys) > 0:    ax1.scatter(buys['trade_date'], buys['close'], marker='^', color='red', s=100, zorder=5, label=f'买入({len(buys)})')if len(sells) > 0:    ax1.scatter(sells['trade_date'], sells['close'], marker='v', color='green', s=100, zorder=5, label=f'卖出({len(sells)})')ax1.legend(ncol=3); ax1.grid(True, alpha=0.3); ax1.set_ylabel('价格')nav = recent['nav'] / 100000bh = recent['close'] / recent['close'].iloc[0]ax2.plot(dates, nav, 'red', lw=1, label='海龟策略净值')ax2.plot(dates, bh, 'gray', ls='--', lw=0.8, label='买入持有净值')ax2.axhline(y=1, color='black', ls=':', lw=0.5)ax2.legend(); ax2.grid(True, alpha=0.3); ax2.set_ylabel('净值')plt.tight_layout(); plt.show()

In [ ]:
# ── 6) 量化指标计算 ──def calc_metrics(df, account_value=100000, risk_free=0.02):    df = df.dropna(subset=['nav'])    final = df['nav'].iloc[-1]    cum_ret = (final - account_value) / account_value * 100    days = len(df)    ann_ret = ((1 + cum_ret/100) ** (252/days) - 1) * 100        peak = df['nav'].cummax()    mdd = ((df['nav'] - peak) / peak * 100).min()        rets = df['nav'].pct_change().dropna()    sharpe = (rets.mean() * 252 - risk_free) / (rets.std() * np.sqrt(252)) if rets.std() > 0 else 0        bh_ret = (df['close'].iloc[-1] / df['close'].iloc[0] - 1) * 100        exit_trades = [t for t in trades if t['type'] in ('止损', '通道退出')]    print(f"━━━━━━━━━━━━ 回测指标 ━━━━━━━━━━━━")    print(f"  累计回报:   {cum_ret:>8.2f}%")    print(f"  年化收益率: {ann_ret:>8.2f}%")    print(f"  最大回撤:   {mdd:>8.2f}%")    print(f"  夏普比率:   {sharpe:>8.4f}")    print(f"  总交易次数: {len(exit_trades):>6}")    print(f"  买入持有:   {bh_ret:>8.2f}%")calc_metrics(df_result)

## 4. 多场景对比实验对不同股票和通道周期组合进行系统对比。

In [ ]:
symbols = ['002594.SZ', '600900.SH', '688981.SH', '688099.SH', '00981.HK', '01211.HK']names = ['比亚迪A', '长江电力', '中芯国际A', '晶晨股份', '中芯国际H', '比亚迪H']combos = [(10,10,'短周期'), (20,10,'系统一'), (55,20,'系统二')]results = []for sym, name in zip(symbols, names):    df = load_data(sym)    for ch, ex, label in combos:        try:            df_r, tr = run_backtest(df, ch, ex)            df_r = df_r.dropna(subset=['nav'])            final = df_r['nav'].iloc[-1]            cr = (final - 100000) / 100000 * 100            peak = df_r['nav'].cummax()            mdd = ((df_r['nav'] - peak) / peak * 100).min()            rets = df_r['nav'].pct_change().dropna()            sharpe = (rets.mean()*252 - 0.02) / (rets.std()*np.sqrt(252)) if rets.std()>0 else 0            exits = [t for t in tr if t['type'] in ('止损','通道退出')]            results.append([name, label, round(cr,2), round(mdd,2), round(sharpe,4), len(exits)])        except Exception as e:            results.append([name, label, 0, 0, 0, f'ERR:{e}'])res_df = pd.DataFrame(results, columns=['股票','策略','累计回报%','最大回撤%','夏普比率','交易次数'])display(res_df)

## 5. 总结与心得1. **趋势股表现优秀**：比亚迪A/H、中芯国际H 在海龟策略下获得了最高 242% 的累计回报2. **震荡/防御股不适合**：长江电力、晶晨股份 的线性趋势不明显，海龟策略频繁止损3. **系统二更稳定**：长周期通道（55/20）在大多数股票上夏普比率更高4. **金字塔加仓**在强趋势中有显著贡献5. **严格止损**是海龟策略的生命线